# BERT RAG with vectorDB
 Requirements:
 * Python 3x. (latest version| https://www.python.org/downloads/release/python-3145/)
 * pgvector database (https://github.com/pgvector/pgvector#installation)
 
 Notes: 
   - Using langchain to ingest and manage retrieval
   - BERT to convert text in to embeddings
   - PGVector to store embeddings
   - BERT


In [2]:
#install dependencies 
%pip install -U langchain langchain-postgres langchain-huggingface langchain-text-splitters sentence-transformers
%pip install -U pgvector psycopg2-binary transformers
%pip install -U langchain_community pypdf

  Using cached pgvector-0.3.6-py3-none-any.whl.metadata (13 kB)
Using cached pgvector-0.3.6-py3-none-any.whl (24 kB)
  Attempting uninstall: pgvector
    Found existing installation: pgvector 0.4.2
    Uninstalling pgvector-0.4.2:
      Successfully uninstalled pgvector-0.4.2

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pgvector-0.4.2-py3-none-any.whl.metadata (19 kB)
Using cached pgvector-0.4.2-py3-none-any.whl (27 kB)
  Attempting uninstall: pgvector
    Found existing installation: pgvector 0.3.6
    Uninstalling pgvector-0.3.6:
      Successfully uninstalled pgvector-0.3.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-postgres 0.0.17 requires pgvector<0.4,>=0.2.5, but you have pgvector 0.4.2 which is

In [ ]:
from langchain_postgres.vectorstores import PGVector
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# initialize sentence transformer 
# all-MiniLM is a popular BERT based Sentence transformer.
# see https://huggingface.co/sentence-transformers
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# connect to pg database
host='localhost'
port='5432' # standard pg port
dbname='embeddingsdb'
user='postgres'
pwd='postgres'

connection = f"postgresql+psycopg://{user}:{pwd}@{host}:{port}/{dbname}"
collection_name = "knowledge_base"


# intialize and load documents to vector store
vector_store = PGVector(connection=connection, 
                        collection_name=collection_name, 
                        embeddings=embeddings,
                        use_jsonb=True
                        # set other params like embedding_lenght for efficiency
                        )

def vectorize(text):
    """
     Vectorizes the given text and stores it in the vector DB.
    """
    if len(text) < 1 :
        print('Empty Text')
        return
    # initailize Text Splitter 
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,       # BERT optimal token window size
        chunk_overlap=50      # Context preservation between chunks
    )

    # Split text to chunks 
    docs = text_splitter.split_documents(text)
    return vector_store.add_documents(docs)

def search(query):
    results = vector_store.similarity_search(query)
    for result in result:
        print(f" - {result.page_content}")
    



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:

# Test with pdf document using langchain_community pdfloader
files = ["2021-bender-parrots.pdf","Elegoo Super Starter Kit for UNO V1.0.2019.09.17.pdf"]

for file in files:
    loader = PyPDFLoader(file) 
    document = loader.load()
    vectorize(document)


In [ ]:
# Simple document search
query = "Find documents by Timit Gebru"
print(f" Question: {query}  \n Answer {vector_store.similarity_search(query)}")

print(f"-----------------------------------")

query = "Find power settings for adrino uno"
print(f" Question: {query}  \n Answer {vector_store.similarity_search(query)}")


 Question: Find documents by Timit Gebru  
 Answer [Document(id='d4275843-8385-4789-9ccc-c00241982151', metadata={'page': 3, 'title': 'On the Dangers of Stochastic Parrots: Can Language Models Be Too Big? "1F99C', 'author': 'Emily M. Bender; Timnit Gebru; Angelina McMillan-Major; Shmargaret Shmitchell', 'source': '2021-bender-parrots.pdf', 'creator': 'LaTeX with acmart 2020/11/15 v1.75 Typesetting articles for the Association for Computing Machinery and hyperref 2020-05-15 v7.00e Hypertext links for LaTeX', 'moddate': '2026-05-18T21:51:57-04:00', 'subject': '-  Computing methodologies  ->  Natural language processing.', 'trapped': '/False', 'keywords': '', 'producer': 'LuaHBTeX, Version 1.12.0 (TeX Live 2020); modified using iText® 7.1.11-SNAPSHOT ©2000-2020 iText Group NV (AGPL-version)', 'page_label': '4', 'pdfversion': '1.5', 'total_pages': 14, 'creationdate': '2021-02-09T18:04:12+00:00', 'ptex.fullbanner': 'This is LuaHBTeX, Version 1.12.0 (TeX Live 2020)'}, page_content='9https://

AttributeError: 'list' object has no attribute 'page_content'